# EDA – Predicción de Churn · AndesLink Servicios Digitales S.A.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

df = pd.read_csv("../data/raw/churn_sintetico.csv")
print(f"Shape: {df.shape}")
df.head()

## 1. Visión general del dataset

In [ ]:
print("=== Tipos de datos ===")
print(df.dtypes)
print()
print("=== Valores nulos ===")
print(df.isnull().sum())
print()
print("=== Estadísticas descriptivas ===")
df.describe()

## 2. Distribución del target `churn`
El target es binario: **1 = cliente que abandona**, **0 = cliente que permanece**.  
Un desbalance moderado puede requerir ajuste de threshold o uso de `class_weight='balanced'`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df["churn"].value_counts()
axes[0].bar(["No churn (0)", "Churn (1)"], counts.values, color=["#4C9BE8", "#E87B4C"])
axes[0].set_title("Distribución del target")
axes[0].set_ylabel("Cantidad de clientes")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, f"{v} ({v/len(df)*100:.1f}%)", ha="center", fontweight="bold")

axes[1].pie(counts.values, labels=["No churn", "Churn"], autopct="%1.1f%%",
            colors=["#4C9BE8", "#E87B4C"], startangle=90)
axes[1].set_title("Proporción churn vs no churn")

plt.suptitle("Variable objetivo: Churn", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/01_target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Ratio desbalance: {counts[0]/counts[1]:.2f}:1")

## 3. Variables numéricas
Distribuciones separadas por clase para identificar features con mayor poder discriminativo.

In [ ]:
num_cols = ["tenure_months", "monthly_charge", "total_charges",
            "support_tickets", "late_payments", "avg_monthly_usage_gb",
            "num_products", "customer_age"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color in [(0, "#4C9BE8"), (1, "#E87B4C")]:
        axes[i].hist(df[df["churn"] == label][col], bins=30, alpha=0.6,
                     color=color, label=f"churn={label}", density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle("Distribución de variables numéricas por clase", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/02_numeric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Variables categóricas – Tasa de churn por categoría

In [ ]:
cat_cols = ["contract_type", "payment_method", "internet_service", "region"]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)["churn"].mean().sort_values(ascending=False)
    bars = axes[i].bar(churn_rate.index, churn_rate.values, color="#E87B4C", alpha=0.8)
    axes[i].set_title(f"Churn rate por {col}")
    axes[i].set_ylabel("Tasa de churn")
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis="x", rotation=30)
    for bar, val in zip(bars, churn_rate.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, val + 0.01,
                     f"{val:.2%}", ha="center", fontsize=9)

plt.suptitle("Tasa de churn por variable categórica", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/03_churn_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Correlación entre variables numéricas

In [ ]:
corr_cols = num_cols + ["churn"]
corr = df[corr_cols].corr()

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, center=0, linewidths=0.5)
plt.title("Matriz de correlación (variables numéricas + churn)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/04_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nCorrelación con target 'churn' (ordenada):")
print(corr["churn"].drop("churn").sort_values(key=abs, ascending=False).to_string())

## 6. Boxplots: features clave por clase

In [ ]:
key_features = ["tenure_months", "monthly_charge", "support_tickets", "late_payments"]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for i, col in enumerate(key_features):
    df.boxplot(column=col, by="churn", ax=axes[i], 
               boxprops=dict(color="#333"), medianprops=dict(color="red", linewidth=2))
    axes[i].set_title(col)
    axes[i].set_xlabel("Churn (0=No, 1=Sí)")

plt.suptitle("Distribución de features clave por clase", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/05_boxplots_key_features.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Hallazgos y conclusiones del EDA

### Variables con mayor poder predictivo (hipótesis inicial):
- **`contract_type`**: clientes con contrato mensual tienen tasa de churn significativamente mayor.
- **`tenure_months`**: clientes con menor antigüedad abandonan más. Es uno de los predictores clásicos.
- **`late_payments`**: más pagos tardíos → mayor probabilidad de churn.
- **`support_tickets`**: alta cantidad de tickets puede indicar insatisfacción.
- **`monthly_charge`**: precios más altos correlacionan con mayor churn.

### Calidad de datos:
- No hay valores nulos → no se requiere imputación.
- Variables categóricas con cardinalidad baja → One-Hot Encoding directo.
- Variables binarias ya codificadas como 0/1.

### Consideraciones para el modelado:
- Desbalance leve (66/34): usar `class_weight='balanced'` o ajustar threshold.
- `total_charges` puede ser colineal con `tenure_months × monthly_charge` → evaluar.
- Escalar variables numéricas continuas para modelos sensibles a escala (Logistic Regression, SVM).
